# R26-DS-004 — preprocess once (Colab)

**Goal:** verify uploads, build `text_primary`, save **Parquet** caches so you do **not** re-run text cleaning for every model experiment.

**Next notebook:** e.g. `tf_idf_logistic_reg.ipynb` (or repo `02_train_baseline_sklearn.ipynb`) loads these Parquets and trains the baseline.

Run cells **top to bottom**. **Cell 1** connects Google Drive on Colab if needed, then sets paths under `Research Project Files`.

In [ ]:
# === CELL 1 — paths (Colab: connects Drive here if not already mounted) ===
from pathlib import Path


def _ensure_colab_drive_mounted() -> None:
    try:
        from google.colab import drive
    except ImportError:
        return
    if Path("/content/drive/MyDrive").is_dir():
        return
    print("Connecting to Google Drive… approve the browser prompt if it appears.")
    drive.mount("/content/drive")


def _drive_root() -> Path:
    for candidate in (Path("/content/drive/MyDrive"), Path("/content/drive/My drive")):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Drive not found at /content/drive/MyDrive. Complete the mount, then re-run this cell."
    )


_ensure_colab_drive_mounted()
DRIVE_ROOT = _drive_root()
RESEARCH_ROOT = DRIVE_ROOT / "Research Project Files"
DATA_ROOT = RESEARCH_ROOT / "transaction_semantic_data"

TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV = DATA_ROOT / "val.csv"
TEST_CSV = DATA_ROOT / "test.csv"

# Preprocessed Parquet + manifest (same Drive folder tree; survives new runtimes)
CACHE_ROOT = RESEARCH_ROOT / "transaction_semantic_cache"
TRAIN_PQ = CACHE_ROOT / "train_preprocessed.parquet"
VAL_PQ = CACHE_ROOT / "val_preprocessed.parquet"
TEST_PQ = CACHE_ROOT / "test_preprocessed.parquet"
MANIFEST_JSON = CACHE_ROOT / "preprocess_manifest.json"

print("DRIVE_ROOT (resolved) =", DRIVE_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("TRAIN_CSV =", TRAIN_CSV)
print("VAL_CSV   =", VAL_CSV)
print("TEST_CSV  =", TEST_CSV)
print("CACHE_ROOT=", CACHE_ROOT)

In [ ]:
# === CELL 2 — verify uploads (prints OK or raises) ===
missing = [p for p in (TRAIN_CSV, VAL_CSV, TEST_CSV) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing CSV(s). Re-run Cell 1 (mounts Drive on Colab if needed), or fix DATA_ROOT in Cell 1:\n"
        + "\n".join(str(p) for p in missing)
    )
print("OK — all three CSVs found.")
for p in (TRAIN_CSV, VAL_CSV, TEST_CSV):
    print(p.name, p.stat().st_size, "bytes")

In [ ]:
# === CELL 3 — installs (run once per fresh runtime) ===
%pip install -q pandas pyarrow

In [ ]:
# === CELL 4 — imports for preprocessing only ===
import json
import re
import unicodedata

import pandas as pd

print("OK — imports")

In [ ]:
# === CELL 5 — text cleaning helpers (no ML here) ===

def normalize_unicode(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    return unicodedata.normalize("NFKC", s)


def clean_description(s: str) -> str:
    s = normalize_unicode(s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def resolve_label(row: pd.Series) -> str:
    """Align with split_labeled_dataset: final_label if present, else reviewer, else suggested."""
    for key in ("final_label", "reviewer_label", "suggested_label"):
        v = row.get(key)
        if pd.notna(v) and str(v).strip():
            return str(v).strip()
    return ""


def build_text_primary(row: pd.Series) -> str:
    """Single string for bag-of-words / TF-IDF; keep tabular fields as bracket tags."""
    parts = [clean_description(row.get("description", ""))]
    bank = row.get("bank_detected")
    if pd.notna(bank) and str(bank).strip():
        parts.append(f"[bank:{str(bank).strip().lower()}]")
    direction = row.get("direction")
    if pd.notna(direction) and str(direction).strip():
        parts.append(f"[dir:{str(direction).strip().upper()}]")
    band = row.get("amount_band")
    if pd.notna(band) and str(band).strip():
        parts.append(f"[band:{str(band).strip().lower()}]")
    doc_type = row.get("document_type")
    if pd.notna(doc_type) and str(doc_type).strip():
        parts.append(f"[doc:{str(doc_type).strip().lower()}]")
    return " ".join(p for p in parts if p)


print("OK — helpers defined")

In [ ]:
# === CELL 6 — load CSVs, attach label + text_primary, drop unlabeled rows ===

def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["label"] = df.apply(resolve_label, axis=1)
    before = len(df)
    df = df[df["label"] != ""].copy()
    df["text_primary"] = df.apply(build_text_primary, axis=1)
    df = df[df["text_primary"].str.len() >= 3]
    print(path.name, "rows", before, "->", len(df), "after label+text filter")
    return df


train_df = load_split(TRAIN_CSV)
val_df = load_split(VAL_CSV)
test_df = load_split(TEST_CSV)

print("OK — dataframes ready")
print("label counts (train):\n", train_df["label"].value_counts())

In [ ]:
# === CELL 7 — choose columns to cache (edit if you add features) ===
CACHE_COLUMNS = [
    "row_id",
    "document_id",
    "filename",
    "description",
    "text_primary",
    "label",
    "amount_lkr",
    "amount_band",
    "direction",
    "bank_detected",
    "document_type",
    "parse_confidence",
    "act_reference",
    "IRD_taxability",
]

existing = set(train_df.columns)
use_cols = [c for c in CACHE_COLUMNS if c in existing]
missing_cols = [c for c in CACHE_COLUMNS if c not in existing]
if missing_cols:
    print("Note: skipping missing columns:", missing_cols)

train_out = train_df[use_cols].copy()
val_out = val_df[use_cols].copy()
test_out = test_df[use_cols].copy()

print("OK — output columns:", use_cols)

In [ ]:
# === CELL 8 — write Parquet + manifest (re-use for any later model) ===
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

train_out.to_parquet(TRAIN_PQ, index=False)
val_out.to_parquet(VAL_PQ, index=False)
test_out.to_parquet(TEST_PQ, index=False)

manifest = {
    "train_rows": len(train_out),
    "val_rows": len(val_out),
    "test_rows": len(test_out),
    "columns": use_cols,
    "train_parquet": str(TRAIN_PQ),
    "val_parquet": str(VAL_PQ),
    "test_parquet": str(TEST_PQ),
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("OK — wrote:")
print(TRAIN_PQ)
print(VAL_PQ)
print(TEST_PQ)
print(MANIFEST_JSON)